# COMPASS — training & evaluation on Kaggle (GPU)

Thin runner: **all logic lives in the git repo**, this notebook only executes it
(`git pull` delivers code updates; never edit code here).

Before running:
- **Accelerator: GPU** (T4 works; ~16 GB)
- **Internet: ON** — needed once to download ImageNet weights (~85 MB).
  If OFF, training still runs but from random init (a warning is printed).
- The `compass-preprocessed` dataset is attached as input data. The setup cell
  finds the data tree **no matter how the mount is nested**, copies it to
  writable space, and automatically applies any **hotfix folders** (e.g.
  `case_01fix/`) that were dropped into the dataset — no manual file surgery.

Stage 1 = 150 epochs, AdamW 1e-4, `L_seg + 0.5·L_contra + 0.3·L_cps`.
The training cell auto-**resumes** from `ckpt_last.pt` if the session was
interrupted — just re-run it.

In [1]:
import os, subprocess

REPO = "https://github.com/Umer-3085/COMPASS.git"
DIR = "/kaggle/working/COMPASS"
if not os.path.exists(DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO, DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=DIR, check=True)
os.chdir(DIR)
print("repo ready at", os.getcwd())

Cloning into '/kaggle/working/COMPASS'...


repo ready at /kaggle/working/COMPASS


In [5]:
subprocess.run(["git", "pull"], cwd="/kaggle/working/COMPASS", check=True)

From https://github.com/Umer-3085/COMPASS
   aa195ec..44e8db6  main       -> origin/main


Updating aa195ec..44e8db6
Fast-forward
 src/model/fusion.py | 10 +++++++---
 src/train.py        |  6 ++++++
 train_kaggle.ipynb  |  6 +++---
 3 files changed, 16 insertions(+), 6 deletions(-)


CompletedProcess(args=['git', 'pull'], returncode=0)

In [2]:
import torch, os
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU!)")

# DEBUG: test discover directly
from pathlib import Path
inputs = Path("/kaggle/input")
TEMPLATE_REL = Path("template") / "ct_template.nii.gz"

hits = sorted(inputs.rglob(TEMPLATE_REL.name),
              key=lambda p: (len(p.parts), str(p)))
hits = [h for h in hits if h.parent.name == "template"]
print("hits:", hits)
if hits:
    root = hits[0].parent.parent
    print("data root:", root)
    print("template exists:", (root / TEMPLATE_REL).exists())
else:
    print("NO HITS - searching for template directories...")
    for d in inputs.rglob("template"):
        if d.is_dir():
            print("  found template dir:", d)
            files = list(d.glob("ct_template.nii.gz"))
            print("    ct_template.nii.gz:", files)

cuda: True Tesla T4
hits: []
NO HITS - searching for template directories...
  found template dir: /kaggle/input/datasets/muhammadumerbsse/compass-preprocessed/compass_preprocessed/preprocessed/template
    ct_template.nii.gz: []
  found template dir: /kaggle/input/datasets/muhammadumerbsse/compass-preprocessed/compass_preprocessed/preprocessed/qc/template
    ct_template.nii.gz: []


In [3]:
# Discover the mounted data (any nesting), copy to /kaggle/working,
# overlay hotfix folders (e.g. case_01fix/), sanity-check 235x235 slices.
import subprocess
subprocess.run(["python", "-m", "src.kaggle_setup"], check=True)
DATA = open("/kaggle/working/compass_data/_data_root.txt").read().strip()
print("DATA =", DATA)

[kaggle_setup] data tree discovered at: /kaggle/input/datasets/muhammadumerbsse/compass-preprocessed/compass_preprocessed/preprocessed
[kaggle_setup] copying /kaggle/input/datasets/muhammadumerbsse/compass-preprocessed/compass_preprocessed/preprocessed -> /kaggle/working/compass_data ...
[kaggle_setup] hotfix folders found: ['case_01fix']
[kaggle_setup] overlaid mr/train/case_01 from case_01fix/
[kaggle_setup] sanity check OK: 40 cases, slices (235, 235)
DATA_ROOT=/kaggle/working/compass_data
DATA = /kaggle/working/compass_data


In [4]:
# Stage 1: 150 epochs. Re-run this cell to resume after interruption.
ckpt = "/kaggle/working/runs/stage1/ckpt_last.pt"
resume = ["--resume"] if os.path.exists(ckpt) else []
subprocess.run([
    "python", "-m", "src.train", "--stage", "1",
    "--data-root", DATA,
    "--out", "/kaggle/working/runs/stage1",
    "--batch-size", "8", "--num-workers", "2", "--amp",
    "--log-every", "50", *resume,
], check=True)

[stage 1] train=2542 slices val=384 slices device=cuda amp=True epochs=150 lr=0.0001
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:04<00:00, 20.6MB/s]


trainable params: 44.6M
  ep 0 it 50: total=8.0078 seg=1.8139 contra=1.3565 cps=18.3855
  ep 0 it 100: total=5.1921 seg=1.4975 contra=1.2776 cps=10.1862
  ep 0 it 150: total=4.0864 seg=1.3022 contra=1.2959 cps=7.1210
  ep 0 it 200: total=3.4563 seg=1.1636 contra=1.2926 cps=5.4879
  ep 0 it 250: total=3.0362 seg=1.0607 contra=1.2664 cps=4.4742
  ep 0 it 300: total=2.7427 seg=0.9804 contra=1.2548 cps=3.7830
epoch 0: total=2.6649 val_dice=0.5472 best=0.5472 (173s) *BEST*
  ep 1 it 50: total=1.2515 seg=0.5272 contra=1.2823 cps=0.2772
  ep 1 it 100: total=1.2169 seg=0.4995 contra=1.2932 cps=0.2359
  ep 1 it 150: total=1.1933 seg=0.4914 contra=1.2740 cps=0.2166
  ep 1 it 200: total=1.1720 seg=0.4828 contra=1.2541 cps=0.2072
  ep 1 it 250: total=1.1638 seg=0.4757 contra=1.2550 cps=0.2021
  ep 1 it 300: total=1.1566 seg=0.4700 contra=1.2577 cps=0.1926
epoch 1: total=1.1473 val_dice=0.5654 best=0.5654 (176s) *BEST*
  ep 2 it 50: total=0.9868 seg=0.4126 contra=1.0753 cps=0.1221
  ep 2 it 100: to

CompletedProcess(args=['python', '-m', 'src.train', '--stage', '1', '--data-root', '/kaggle/working/compass_data', '--out', '/kaggle/working/runs/stage1', '--batch-size', '8', '--num-workers', '2', '--amp', '--log-every', '50'], returncode=0)

**Notes**
- `runs/stage1/train_log.csv` grows one row per epoch; `ckpt_best.pt` tracks
  the best validation mean foreground Dice.
- `/kaggle/working` is wiped when the session ends → after Stage 1 finishes,
  run the download cell at the bottom (or copy checkpoints into a Kaggle
  dataset) before closing.
- Stage 2 (frozen encoders+fusion, new decoder + evidential head,
  75 epochs, AdamW 5e-5) needs the Stage-1 best checkpoint in the same session.

In [18]:
import os, sys
os.chdir("/kaggle/working/COMPASS")  # or wherever your repo cloned
sys.path.insert(0, "/kaggle/working/COMPASS")
print("CWD:", os.getcwd())
print("src exists:", os.path.exists("src"))

CWD: /kaggle/working/COMPASS
src exists: True


In [22]:
import pathlib, re

p = pathlib.Path("/kaggle/working/COMPASS/src/train.py")
txt = p.read_text()

# Find the Stage 2 model setup and ensure decoder moves to device
# Look for where new decoder is attached
if "model.decoder = " in txt and ".to(device)" not in txt.split("model.decoder = ")[1][:200]:
    # Patch: add .to(device) after decoder assignment
    txt = txt.replace(
        "model.decoder = Decoder(",
        "model.decoder = Decoder(\n        ).to(device)  # ensure new decoder on GPU\n        # "
    )
    # Better: find the exact block and fix properly
    # Let's do a more targeted fix
    print("Found decoder assignment, patching...")
    
# Actually, let's just ensure the whole model is on device after Stage 2 setup
# Find "Stage 2" comment area
if "# Stage 2" in txt or "stage 2" in txt.lower():
    # Insert model.to(device) after Stage 2 model modifications
    pass

p.write_text(txt)
print("Patched. Re-run Stage 2 cell.")

Patched. Re-run Stage 2 cell.


In [31]:
# Stage 2 (run after Stage 1 completes in this session)
s1_best = "/kaggle/working/runs/stage1/ckpt_best.pt"
assert os.path.exists(s1_best), "finish Stage 1 first"
ckpt2 = "/kaggle/working/runs/stage2/ckpt_last.pt"
resume2 = ["--resume"] if os.path.exists(ckpt2) else []
subprocess.run([
    "python", "-m", "src.train", "--stage", "2",
    "--init-from", s1_best,
    "--data-root", DATA,
    "--out", "/kaggle/working/runs/stage2",
    "--batch-size", "8", "--num-workers", "2", "--amp",
    "--log-every", "50", *resume2,
], check=True)

[stage 2] train=2542 slices val=384 slices device=cuda amp=True epochs=75 lr=5e-05
  [stage2] encoders+fusion frozen; new decoder attached
trainable params: 2.1M
  ep 0 it 50: total=2.9516 seg=2.0491 evidential=0.9026 dir_nll=0.9026 kl=0.4136 lam=0.0000
  ep 0 it 100: total=2.6497 seg=1.7606 evidential=0.8890 dir_nll=0.8890 kl=0.4005 lam=0.0000
  ep 0 it 150: total=2.4347 seg=1.5604 evidential=0.8743 dir_nll=0.8743 kl=0.3919 lam=0.0000
  ep 0 it 200: total=2.2711 seg=1.4119 evidential=0.8592 dir_nll=0.8592 kl=0.3892 lam=0.0000
  ep 0 it 250: total=2.1443 seg=1.2994 evidential=0.8449 dir_nll=0.8449 kl=0.3913 lam=0.0000
  ep 0 it 300: total=2.0361 seg=1.2063 evidential=0.8298 dir_nll=0.8298 kl=0.3983 lam=0.0000
epoch 0: total=2.0033 val_dice=0.6452 best=0.6452 (121s) *BEST*
  ep 1 it 50: total=1.3511 seg=0.6336 evidential=0.7175 dir_nll=0.7151 kl=0.4942 lam=0.0050
  ep 1 it 100: total=1.3121 seg=0.6013 evidential=0.7108 dir_nll=0.7082 kl=0.5110 lam=0.0050
  ep 1 it 150: total=1.2698 seg=

CompletedProcess(args=['python', '-m', 'src.train', '--stage', '2', '--init-from', '/kaggle/working/runs/stage1/ckpt_best.pt', '--data-root', '/kaggle/working/compass_data', '--out', '/kaggle/working/runs/stage2', '--batch-size', '8', '--num-workers', '2', '--amp', '--log-every', '50'], returncode=0)

In [33]:
# Six-condition evaluation (uses the Stage-2 best checkpoint)
s2_best = "/kaggle/working/runs/stage2/ckpt_best.pt"
assert os.path.exists(s2_best), "finish Stage 2 first"
subprocess.run([
    "python", "-m", "src.evaluate",
    "--ckpt", s2_best,
    "--data-root", DATA,
    "--out", "/kaggle/working/runs/eval",
    "--batch-size", "8", "--num-workers", "2",
], check=True)

checkpoint: epoch 40, stage 2, best val dice 0.8848, UG probs for calibration: True
test set: 768 slices

=== condition: Full (1, 1, 1) ===
  MR_10: pix_acc=0.9065 ece=0.2051 brier=3.1652 vacuity=0.4952
    class 1: dice=0.5878 iou=0.4162 prec=0.5631 rec=0.6147 hd95=47.03
    class 2: dice=0.0000 iou=0.0000 prec=0.0000 rec=0.0000 hd95=356.28
    class 3: dice=0.2174 iou=0.1220 prec=0.4504 rec=0.1433 hd95=36.85
    class 4: dice=0.5823 iou=0.4107 prec=0.4796 rec=0.7409 hd95=227.28
  MR_33: pix_acc=0.9578 ece=0.2207 brier=3.4663 vacuity=0.4627
    class 1: dice=0.8082 iou=0.6781 prec=0.7759 rec=0.8433 hd95=21.64
    class 2: dice=0.7876 iou=0.6497 prec=0.8974 rec=0.7018 hd95=196.73
    class 3: dice=0.8180 iou=0.6920 prec=0.8465 rec=0.7913 hd95=132.70
    class 4: dice=0.9263 iou=0.8627 prec=0.8698 rec=0.9906 hd95=4.95
  MR_39: pix_acc=0.9721 ece=0.2407 brier=2.1626 vacuity=0.4849
    class 1: dice=0.8711 iou=0.7716 prec=0.8014 rec=0.9541 hd95=40.93
    class 2: dice=0.9277 iou=0.8652 pr

CompletedProcess(args=['python', '-m', 'src.evaluate', '--ckpt', '/kaggle/working/runs/stage2/ckpt_best.pt', '--data-root', '/kaggle/working/compass_data', '--out', '/kaggle/working/runs/eval', '--batch-size', '8', '--num-workers', '2'], returncode=0)

In [32]:
import shutil
from google.colab import files

# Zip Stage 2 checkpoints
shutil.make_archive("/kaggle/working/stage2_checkpoints", "zip", "/kaggle/working/runs/stage2")
print("Created: /kaggle/working/stage2_checkpoints.zip")

# Force browser download (bypasses UI issues)
files.download("/kaggle/working/stage2_checkpoints.zip")

Created: /kaggle/working/stage2_checkpoints.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import shutil
shutil.make_archive("/kaggle/working/eval_results", "zip", "/kaggle/working/runs/eval")

'/kaggle/working/eval_results.zip'

In [38]:
# --------------------------------------------------------------
#  QUICK POST‑HOC ANALYSIS – confusion matrices + calibration curves
#  (uses the same single test loader + per‑condition masking)
# --------------------------------------------------------------
import torch, numpy as np, pandas as pd, json, os, shutil, matplotlib.pyplot as plt
from pathlib import Path
from src.data import make_dataloader
from src.model.compass import COMPASS
from src.losses import uncertainty_guided_probs

device = "cuda"

# ---------- 1. load model (Stage‑2) ----------
model = COMPASS().to(device)
model.init_stage2()                                   # builds decoder2
ckpt = torch.load("/kaggle/working/runs/stage2/ckpt_best.pt",
                  map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

# ---------- 2. single test loader (full modalities) ----------
DATA_ROOT   = Path("/kaggle/working/compass_data")
template    = DATA_ROOT / "template" / "ct_template.nii.gz"
loader = make_dataloader(
    DATA_ROOT, "test", template,
    batch_size=8, augment=False, num_workers=2
)
print(f"test set: {len(loader.dataset)} slices")

# ---------- 3. six conditions ----------
CONDITIONS = {
    "Full"    : (1,1,1),
    "CT+T1"   : (1,1,0),
    "CT+T2"   : (1,0,1),
    "T1-only" : (0,1,0),
    "T2-only" : (0,0,1),
    "CT-only" : (1,0,0),
}

out_dir = Path("/kaggle/working/runs/eval/confusion_matrices")
out_dir.mkdir(parents=True, exist_ok=True)

all_cm = {}
calib_data = []

# ---------- helper ----------
def cm_per_class(pred, target, n_cls=5):
    cm = np.zeros((n_cls, n_cls), dtype=np.int64)
    for c in range(n_cls):
        pc = (pred == c)
        for t in range(n_cls):
            cm[c, t] = (pc & (target == t)).sum()
    return cm

with torch.no_grad():
    for name, cond in CONDITIONS.items():
        cm_tot = np.zeros((5,5), dtype=np.int64)
        probs_all, targ_all = [], []

        # build mask tensor once (will be broadcast per batch)
        mask_tensor = torch.tensor([cond], dtype=torch.float32, device=device)

        for batch in loader:
            ct = batch["ct"].to(device)
            t1 = batch["t1"].to(device)
            t2 = batch["t2"].to(device)
            lab = batch["label"].to(device)          # (B,H,W)
            B = ct.shape[0]

            # repeat mask for batch
            mask = mask_tensor.repeat(B, 1)           # (B,3)

            # forward – model uses mask to zero missing modalities
            out = model(ct, t1, t2, mask)
            probs = uncertainty_guided_probs(out["seg_logits"], out["evidence"], enabled=True)
            probs = probs.cpu()                       # (B,5,H,W)
            pred  = probs.argmax(dim=1).numpy()       # (B,H,W)
            targ  = lab.cpu().numpy()

            for b in range(B):
                cm_tot += cm_per_class(pred[b], targ[b])

                # calibration data (voxel‑wise)
                conf = probs[b].max(0).values.numpy().ravel()
                corr = (pred[b] == targ[b]).astype(np.float32).ravel()
                probs_all.append(conf)
                targ_all.append(corr)

        # ---- save confusion matrix ----
        np.save(out_dir / f"cm_{name}.npy", cm_tot)
        pd.DataFrame(cm_tot,
                     index=[f"pred_{i}" for i in range(5)],
                     columns=[f"true_{i}" for i in range(5)]
                    ).to_csv(out_dir / f"cm_{name}.csv")
        all_cm[name] = cm_tot

        # ---- calibration curve (10 bins) ----
        conf = np.concatenate(probs_all)
        corr = np.concatenate(targ_all)
        bins = np.linspace(0,1,11)
        bin_idx = np.digitize(conf, bins) - 1
        bin_acc = [corr[bin_idx==b].mean() if (bin_idx==b).any() else np.nan for b in range(10)]
        bin_conf = [conf[bin_idx==b].mean() if (bin_idx==b).any() else np.nan for b in range(10)]
        calib_data.append(pd.DataFrame({
            "condition": name,
            "bin_center": bin_conf,
            "accuracy"  : bin_acc
        }))

# ---------- 4. tidy CSVs + plot ----------
pd.concat(calib_data).to_csv(out_dir / "calibration_curves.csv", index=False)

plt.figure(figsize=(6,5))
for name in CONDITIONS:
    df = pd.read_csv(out_dir / "calibration_curves.csv")
    df = df[df.condition==name]
    plt.plot(df.bin_center, df.accuracy, marker='o', label=name)
plt.plot([0,1],[0,1],'k--',alpha=0.3)
plt.xlabel("Mean predicted confidence")
plt.ylabel("Empirical accuracy")
plt.title("Calibration curves – 6 modality conditions")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(out_dir / "calibration_curves.png", dpi=200)
plt.close()

print("✅  Confusion matrices + calibration curves written to", out_dir)

test set: 768 slices
✅  Confusion matrices + calibration curves written to /kaggle/working/runs/eval/confusion_matrices


In [39]:
import shutil
shutil.make_archive(
    "/kaggle/working/eval_results_full",   # output path without .zip
    "zip",
    "/kaggle/working/runs/eval"
)
print("Created /kaggle/working/eval_results_full.zip")

Created /kaggle/working/eval_results_full.zip
